# Figure 8 — More than two populations

Pena, Vellmer, Bernardi, Roque & Lindner (2018), Front. Comput. Neurosci. 12:9, doi:10.3389/fncom.2018.00009. Scheme after Dummer, Wieland & Lindner (2014).

**Caption (paper):** (A) RS, LTS, FS: N_LTS = 0.25 N_E, C_LTS = 0.25 C_E, N_FS = 0.05 N_E, C_FS = 0.05 C_E; g = 4; τRS = 20, τLTS = 19, τFS = 21 ms (caption; the text lists 19/20/21 for RS/LTS/FS); RS→FS weight ×1.4. (B) Two modules of 1.25×10⁵ cells, τ = 20 ms, J = 0.1 mV (module 1) and 0.27 mV (module 2), g = 4; module 2 takes half of its E and I inputs from module 1 and shows a hump near 80 Hz.

**Method here:** one scheme neuron per population (3 for A, 4 for B, all-past averaging); networks with N_E = 10,000 (A) and 10,000 per module (B) instead of 100,000. Caption time constants used. Weights into module 2 use J = 0.27 mV.

**Shared implementation choices** (all figures, see `pena_scheme.py`, `pena_network.py`):
- Euler, `dt = 0.1 ms`; voltage clamped at reset during the refractory period (scheme and network).
- Spectra: Eq. 5, two-sided, 2 s windows after a 1 s transient (Δf = 0.5 Hz).
  Fano factor = mean of S at 0.5 and 1 Hz divided by ν; τc from Eq. 7 with the estimator variance subtracted.
- Scheme generation 0: white noise at the diffusion-approximation (mean-field) rate; the paper does not
  specify its starting point. Inhibition-dominated cases (g > 4) use the all-past rate average (Eq. 11).
- Networks are simulated at reduced size (default N_E = 10,000 instead of 100,000) to fit 2 CPUs / ~2 GB;
  partners are drawn with replacement. Larger sizes are available via the config cell but need far more memory.

In [1]:
from pathlib import Path
import sys, json, time
import numpy as np
import matplotlib.pyplot as plt

HERE = Path.cwd()
if not (HERE / "pena2018_params.py").exists():
    HERE = HERE / "05_Pena2018_Replication"
sys.path.insert(0, str(HERE))

from pena2018_params import params_for
import pena_scheme as ps
import pena_runs as pr
FIGURES = HERE / "figures"; FIGURES.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

In [2]:
P = params_for(8)
N_E_NET, N_WINDOWS = 10_000, 10

In [ ]:
namesA, tausA, CA, WA, popsA, connsA = pr.three_pop_spec(P, N_E_NET)
schA = pr.population_scheme("threepop", tausA, P["RI_ext"], CA, WA, P)
netA = pr.multi_network("threepop", popsA, connsA, P, N_WINDOWS)
namesB, tausB, CB, WB, popsB, connsB = pr.module_spec(P, N_E_NET)
schB = pr.population_scheme("modules", tausB, P["RI_ext"], CB, WB, P)
netB = pr.multi_network("modules", popsB, connsB, P, N_WINDOWS)

gen  0:    6.03->   5.61 Hz     8.45->   7.93 Hz    23.32->  22.60 Hz
gen  1:    5.61->   5.72 Hz     7.93->   8.46 Hz    22.60->  23.09 Hz
gen  2:    5.72->   2.54 Hz     8.46->   4.73 Hz    23.09->  19.00 Hz
gen  3:    4.13->   4.19 Hz     6.60->   7.09 Hz    21.05->  17.29 Hz
gen  4:    4.15->   6.71 Hz     6.76->   9.80 Hz    19.79->  19.47 Hz
gen  5:    4.79->   7.80 Hz     7.52->  10.80 Hz    19.71->  22.20 Hz
gen  6:    5.39->   7.28 Hz     8.18->  10.26 Hz    20.21->  23.51 Hz
gen  7:    5.71->   5.97 Hz     8.52->   8.80 Hz    20.76->  23.19 Hz
gen  8:    5.74->   4.63 Hz     8.56->   7.38 Hz    21.11->  22.13 Hz
gen  9:    5.60->   3.59 Hz     8.42->   6.15 Hz    21.23->  20.78 Hz
gen 10:    5.38->   3.00 Hz     8.16->   5.49 Hz    21.18->  19.61 Hz
gen 11:    5.14->   2.87 Hz     7.90->   5.37 Hz    21.03->  18.83 Hz
gen 12:    4.94->   3.20 Hz     7.67->   5.67 Hz    20.83->  18.61 Hz
gen 13:    4.79->   3.73 Hz     7.50->   6.43 Hz    20.64->  18.71 Hz
gen 14:    4.71->   

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
for ax, names, s, n, t in [(axes[0], namesA, schA, netA, "A  RS / LTS / FS"),
                           (axes[1], namesB, schB, netB, "B  two modules")]:
    cols = plt.cm.tab10(np.arange(len(names)))
    for a, (nm, c) in enumerate(zip(names, cols)):
        ax.plot(n["f"][1:], n[f"S_{nm}"][1:], "-", color=c, lw=1, alpha=0.7, label=f"{nm} network")
        ax.plot(s["f"][1:], s["S_final"][a][1:], "--", color=c, lw=1.5, label=f"{nm} scheme")
    ax.set(xlim=(0, 300), xlabel="f (Hz)", ylabel="S(f) (Hz)", title=t)
    ax.legend(frameon=False, fontsize=7, ncol=2)
fig.tight_layout(); fig.savefig(FIGURES / "figure08_multi_population.png", dpi=200); plt.show()
for names, s, n in [(namesA, schA, netA), (namesB, schB, netB)]:
    for a, nm in enumerate(names):
        fcut = 2 * max(float(n[f"nu_{x}"]) for x in names)
        e = ps.relative_error(s["f"], n[f"S_{nm}"], s["S_final"][a], fcut)
        print(f"{nm:4s}: rate scheme {s['nu_final'][a]:6.2f}  network {float(n[f'nu_{nm}']):6.2f} Hz,  Eq.19 error {e:.3f}")

## Notes

_Filled in after the run._